# 1. What you'll learn

        - see the margin and support vectors that determine an SVM
- compare linear, polynomial, and RBF kernels on identical data
- measure how C, gamma, and feature scale change the boundary

        **The one question this notebook answers:** How can a maximum-margin classifier draw a nonlinear boundary without explicitly inventing every curved feature?

# 2. Setup

We use scikit-learn SVC models because they expose support vectors and decision scores. Every variant sees the same split, so boundary shape—not a lucky partition—is what changes.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

The two moon coordinates are deliberately nonlinear and noisy. The target is which moon generated each point; the balanced classes make accuracy readable here.

**Small example:** For five signed distances [-2, -0.4, 0.1, 0.8, 2], the points near zero constrain the boundary. Moving the far points barely changes the margin; moving -0.4 or 0.1 can.

In [ ]:
X,y=make_moons(n_samples=700,noise=.24,random_state=SEED)
frame=pd.DataFrame(X,columns=["x1","x2"]).assign(class_id=y)
print("Shape:",frame.shape,"class balance:",frame.class_id.value_counts(normalize=True).round(3).to_dict())
print(frame.head(5).round(3))

# 4. Explore

The first view shows curved geometry, the second shows overlap near the boundary, and the third shows how one arbitrary unit change distorts Euclidean distance.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.8))
axes[0].scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",s=16);axes[0].set_title("Two interlocking moons")
sns.histplot(data=frame,x="x1",hue="class_id",element="step",stat="density",common_norm=False,ax=axes[1]);axes[1].set_title("One feature cannot separate classes")
X_units=X.copy();X_units[:,0]*=100;axes[2].scatter(X_units[:,0],X_units[:,1],c=y,cmap="coolwarm",s=16);axes[2].set(title="Same points, x1 in centimetres",xlabel="100 × x1",ylabel="x2")
plt.tight_layout();plt.show()

**Takeaway:** A straight line cannot follow both crescent-shaped classes.

**Takeaway:** Each coordinate overlaps strongly by itself, so the interaction between coordinates matters.

**Takeaway:** Changing a unit stretches the geometry even though it adds no information.

# 5. Prepare

We stratify before fitting and put standardization inside each pipeline. SVC's gamma is a distance scale, so learning that scale from held-out rows would be leakage.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
models={name:make_pipeline(StandardScaler(),SVC(kernel=kernel,C=2,gamma="scale",degree=3)) for name,kernel in [("linear","linear"),("polynomial","poly"),("RBF","rbf")]}
print("Train/test:",X_train.shape,X_test.shape)

# 6. Train

        An SVM maximizes the empty band around its boundary while charging a penalty for margin violations. A kernel supplies similarities as if the data had been mapped into a richer feature space.

        Conceptually, fitting follows these steps:

        1. Standardize coordinates using training statistics.
2. Compute pairwise similarities using the chosen kernel.
3. Find the boundary with the widest margin subject to C-weighted violations.
4. Keep only points with nonzero dual weight: the support vectors.

In [ ]:
for model in models.values(): model.fit(X_train,y_train)
for name,model in models.items():
    svc=model.named_steps["svc"];print(f"{name:10s} support vectors={svc.n_support_.sum():3d}, test accuracy={model.score(X_test,y_test):.3f}")

# 7. Evaluate

The three decision maps make capacity visible. A majority baseline is included, while test accuracy and the number of support vectors quantify fit and complexity.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
x0=np.linspace(X[:,0].min()-.5,X[:,0].max()+.5,180);x1=np.linspace(X[:,1].min()-.5,X[:,1].max()+.5,180);xx,yy=np.meshgrid(x0,x1);grid=np.c_[xx.ravel(),yy.ravel()]
baseline=max(np.mean(y_test),1-np.mean(y_test));fig,axes=plt.subplots(1,3,figsize=(14,3.8))
for ax,(name,model) in zip(axes,models.items()):
    score=model.decision_function(grid).reshape(xx.shape);ax.contourf(xx,yy,score,levels=20,cmap="RdBu",alpha=.65);ax.contour(xx,yy,score,levels=[-1,0,1],colors=["white","black","white"],linewidths=[.8,1.8,.8]);ax.scatter(X_test[:,0],X_test[:,1],c=y_test,cmap="coolwarm",s=14,edgecolor="white",linewidth=.2);ax.set_title(f"{name}: acc {model.score(X_test,y_test):.3f}")
fig.suptitle(f"Same split and budget · majority baseline {baseline:.3f}");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The RBF kernel follows the moons without the rigid shape of a polynomial, while the linear margin underfits; the gain is only meaningful because every model faced the same untouched test rows.

# 8. Break it

We multiply x1 by 1,000 and fit an RBF SVM without scaling. Gamma now interprets almost all pairs as far apart along one arbitrary unit, so neighbourhood geometry and generalization deteriorate.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
X_bad=X.copy();X_bad[:,0]*=1000
Xa,Xb,ya,yb=train_test_split(X_bad,y,test_size=.25,stratify=y,random_state=SEED)
raw=SVC(kernel="rbf",C=2,gamma="scale").fit(Xa,ya);scaled=make_pipeline(StandardScaler(),SVC(kernel="rbf",C=2,gamma="scale")).fit(Xa,ya)
print(f"Unscaled accuracy={raw.score(Xb,yb):.3f}; scaled accuracy={scaled.score(Xb,yb):.3f}; damage={scaled.score(Xb,yb)-raw.score(Xb,yb):+.3f}")

**Use this when…** you have a medium-sized dataset, meaningful distances, and a boundary that may be smooth but nonlinear.

        **Don't use this when…** you have millions of rows, poorly scaled mixed data, need native probabilities, or require a transparent global rule.

        ## Try this

        1. Set C to 0.05 and 100; watch margin violations trade against a tighter boundary.
2. Sweep gamma from 0.05 to 50; expect underfit, a useful middle, then tiny islands.
3. Add 20 noise columns; compare the RBF SVM before and after feature selection.